In [1]:
# IMPORT LIBRARIES & CONNECT

In [2]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('ipl_data.db')
print("Connected to database.")

Connected to database.


In [3]:
# PLAYER OF THE MATCH COUNT (RCB WINS)

In [4]:
query = """
SELECT 
    player_of_match,
    COUNT(*) AS potm_count
FROM matches
WHERE winner LIKE 'Royal Challengers%'
GROUP BY player_of_match
ORDER BY potm_count DESC
"""

potm_counts = pd.read_sql_query(query, conn)
print("Player of the Match awards (RCB wins):")
print(potm_counts.head(10))

Player of the Match awards (RCB wins):
  player_of_match  potm_count
0  AB de Villiers          23
1         V Kohli          17
2        CH Gayle          17
3       JH Kallis           5
4       YS Chahal           4
5      GJ Maxwell           4
6    F du Plessis           4
7   R Vinay Kumar           3
8  Mohammed Siraj           3
9        HV Patel           3


In [5]:
# PLAYERS WITH ABOVE-AVERAGE POTM AWARDS

In [6]:
query = """
SELECT 
    player_of_match,
    COUNT(*) AS potm_count
FROM matches
WHERE winner LIKE 'Royal Challengers%'
GROUP BY player_of_match
HAVING COUNT(*) > (
    SELECT AVG(potm_count)
    FROM (
        SELECT COUNT(*) AS potm_count
        FROM matches
        WHERE winner LIKE 'Royal Challengers%'
        GROUP BY player_of_match
    )
)
ORDER BY potm_count DESC
"""

above_avg_potm = pd.read_sql_query(query, conn)
print("Players with above-average POTM awards:")
print(above_avg_potm)


Players with above-average POTM awards:
  player_of_match  potm_count
0  AB de Villiers          23
1         V Kohli          17
2        CH Gayle          17
3       JH Kallis           5
4       YS Chahal           4
5      GJ Maxwell           4
6    F du Plessis           4


In [7]:
# AVERAGE RCB TEAM SCORE PER MATCH

In [8]:
query = """
SELECT AVG(match_runs) AS avg_team_score
FROM (
    SELECT 
        match_id,
        SUM(total_runs) AS match_runs
    FROM deliveries
    WHERE batting_team LIKE 'Royal Challengers%'
    GROUP BY match_id
)
"""

avg_score = pd.read_sql_query(query, conn)
print(f"Average RCB team score: {avg_score['avg_team_score'][0]:.1f} runs")


Average RCB team score: 159.3 runs


In [9]:
# MATCHES WHERE RCB SCORED ABOVE THEIR AVERAGE

In [10]:
query = """
SELECT 
    d.match_id,
    m.season,
    SUM(d.total_runs) AS match_runs
FROM deliveries d
JOIN matches m ON d.match_id = m.id
WHERE d.batting_team LIKE 'Royal Challengers%'
GROUP BY d.match_id
HAVING SUM(d.total_runs) > (
    SELECT AVG(match_runs)
    FROM (
        SELECT 
            match_id,
            SUM(total_runs) AS match_runs
        FROM deliveries
        WHERE batting_team LIKE 'Royal Challengers%'
        GROUP BY match_id
    )
)
ORDER BY match_runs DESC
LIMIT 10
"""

high_scores = pd.read_sql_query(query, conn)
print("Matches where RCB scored above their average (top 10):")
print(high_scores)

Matches where RCB scored above their average (top 10):
   match_id season  match_runs
0    598027   2013         263
1   1426268   2024         262
2    980987   2016         248
3   1426296   2024         241
4    829795   2015         235
5    980907   2016         227
6    829785   2015         226
7   1426274   2024         221
8   1426306   2024         218
9   1359498   2023         218


In [11]:
# VENUES WITH >50% WIN RATE

In [12]:
query = """
SELECT 
    standardized_venue AS venue,
    COUNT(*) AS total_matches,
    COUNT(CASE WHEN winner LIKE 'Royal Challengers%' THEN 1 END) AS wins,
    COUNT(CASE WHEN winner NOT LIKE 'Royal Challengers%' AND winner IS NOT NULL THEN 1 END) AS losses,
    ROUND(
        COUNT(CASE WHEN winner LIKE 'Royal Challengers%' THEN 1 END) * 100.0 / COUNT(*), 
        1
    ) AS win_pct
FROM (
    SELECT 
        CASE 
            WHEN venue LIKE '%Chinnaswamy%' OR (venue LIKE '%Bengaluru%' AND venue LIKE '%Stadium%') 
                THEN 'M. Chinnaswamy Stadium, Bengaluru'
            WHEN venue LIKE '%Punjab Cricket Association%' OR venue LIKE '%IS Bindra%' 
                THEN 'Punjab Cricket Association IS Bindra Stadium, Mohali'
            WHEN venue LIKE '%Dubai%' THEN 'Dubai International Cricket Stadium'
            WHEN venue LIKE '%Wankhede%' THEN 'Wankhede Stadium, Mumbai'
            WHEN venue LIKE '%Feroz Shah%' OR venue LIKE '%Kotla%' 
                THEN 'Feroz Shah Kotla, Delhi'
            WHEN venue LIKE '%Chidambaram%' OR venue LIKE '%Chepauk%' 
                THEN 'MA Chidambaram Stadium, Chepauk, Chennai'
            ELSE TRIM(venue)
        END AS standardized_venue,
        winner
    FROM matches
    WHERE team1 LIKE 'Royal Challengers%'
       OR team2 LIKE 'Royal Challengers%'
) AS cleaned_venues
GROUP BY standardized_venue
HAVING COUNT(*) >= 3
ORDER BY win_pct DESC, total_matches DESC;
"""

good_venues = pd.read_sql_query(query, conn)
print("Venues where RCB wins >50% of matches:")
print(good_venues)

Venues where RCB wins >50% of matches:
                                                venue  total_matches  wins  \
0                                           Kingsmead              4     3   
1                               New Wanderers Stadium              4     3   
2                             Feroz Shah Kotla, Delhi              7     5   
3                 Dubai International Cricket Stadium             12     8   
4       Maharashtra Cricket Association Stadium, Pune              3     2   
5                                     SuperSport Park              3     2   
6   Punjab Cricket Association IS Bindra Stadium, ...              8     5   
7                  Dr DY Patil Sports Academy, Mumbai              4     2   
8             Maharashtra Cricket Association Stadium              4     2   
9                   M. Chinnaswamy Stadium, Bengaluru             90    44   
10                           Wankhede Stadium, Mumbai             18     8   
11                       

In [13]:
# TOP 5 INDIVIDUAL BATTING PERFORMANCES

In [14]:
query = """
SELECT 
    match_id,
    batter,
    runs_scored
FROM (
    SELECT 
        match_id,
        batter,
        SUM(batsman_runs) AS runs_scored
    FROM deliveries
    WHERE batting_team LIKE 'Royal Challengers%'
    GROUP BY match_id, batter
)
ORDER BY runs_scored DESC
LIMIT 5
"""

top_innings = pd.read_sql_query(query, conn)
print("Top 5 individual batting performances:")
print(top_innings)


Top 5 individual batting performances:
   match_id          batter  runs_scored
0    598027        CH Gayle          175
1    829795  AB de Villiers          133
2    980987  AB de Villiers          129
3    548372        CH Gayle          128
4    829785        CH Gayle          117


In [15]:
# CLOSE CONNECTION

In [16]:
conn.close()
print("Connection closed.")

Connection closed.
